# Heart Disease — Exploratory Data Analysis

**Dataset:** UCI Heart Disease (Cleveland), 14 features, binary target (`0` = no disease, `1` = disease).

**Goal:** Understand the data distribution, missingness, class balance, and feature relationships before modelling.

**Outputs:** All plots are saved to `artifacts/plots/eda/` so they can be embedded in the final report.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import (
    ALL_FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES,
    PROCESSED_CSV, TARGET_COLUMN, load_processed,
)

PLOTS_DIR = PROJECT_ROOT / 'artifacts' / 'plots' / 'eda'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
print(f'Project root: {PROJECT_ROOT}')
print(f'Processed CSV: {PROCESSED_CSV}')

## 1. Load the cleaned dataset

If the CSV is missing, run `python scripts/download_data.py` from the project root first.

In [ ]:
df = load_processed()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').round(2)

## 2. Missing-value audit

We need to know which columns require imputation in the preprocessing pipeline.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct}).query('count > 0')
missing_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
if missing_df.empty:
    ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14)
    ax.axis('off')
else:
    sns.barplot(x=missing_df.index, y=missing_df['pct'], ax=ax, color='#fb5607')
    ax.set_ylabel('Missing %')
    ax.set_title('Missing values by feature')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'missing_values.png')
plt.show()

## 3. Class balance

Stratified splits and models robust to mild imbalance are appropriate here.

In [ ]:
balance = df[TARGET_COLUMN].value_counts(normalize=True).rename('share') * 100
balance.index = balance.index.map({0: 'No disease', 1: 'Disease'})
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=balance.index, y=balance.values, ax=ax, palette=['#3a86ff', '#fb5607'])
ax.set_ylabel('% of patients')
ax.set_title('Target class balance')
for i, v in enumerate(balance.values):
    ax.text(i, v + 0.5, f'{v:0.1f}%', ha='center', fontsize=11)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'class_balance.png')
plt.show()
print(balance.round(1).to_string())

## 4. Numeric feature distributions

Histograms with KDE overlays — split by target class to surface feature-level signal.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, NUMERIC_FEATURES):
    sns.histplot(
        data=df, x=col, hue=TARGET_COLUMN,
        element='step', stat='density', common_norm=False,
        palette=['#3a86ff', '#fb5607'], ax=ax,
    )
    ax.set_title(col)
fig.suptitle('Numeric features by target class', y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'numeric_distributions.png')
plt.show()

## 5. Categorical feature breakdown

Stacked bar charts showing disease prevalence across each categorical level.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, CATEGORICAL_FEATURES):
    pct = (
        df.groupby(col)[TARGET_COLUMN]
        .value_counts(normalize=True)
        .unstack(fill_value=0) * 100
    )
    pct.columns = pct.columns.map({0: 'No disease', 1: 'Disease'})
    pct.plot(kind='bar', stacked=True, ax=ax, color=['#3a86ff', '#fb5607'])
    ax.set_title(col)
    ax.set_ylabel('%')
    ax.legend().remove()
axes.flat[-1].axis('off')  # one extra slot in the 2x4 grid
fig.suptitle('Disease prevalence by categorical feature', y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'categorical_distributions.png')
plt.show()

## 6. Correlation heatmap

Pearson correlation among numeric features and the target. Strong absolute correlations point to the most informative features.

In [ ]:
corr_cols = NUMERIC_FEATURES + [TARGET_COLUMN]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
    square=True, ax=ax, cbar_kws={'shrink': 0.7},
)
ax.set_title('Pearson correlation (numeric features + target)')
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'correlation_heatmap.png')
plt.show()

In [ ]:
(corr[TARGET_COLUMN]
 .drop(TARGET_COLUMN)
 .abs()
 .sort_values(ascending=False)
 .rename('|corr| with target'))

## 7. Boxplots — numeric features vs target

Visual confirmation of the correlation findings.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, NUMERIC_FEATURES):
    sns.boxplot(
        data=df, x=TARGET_COLUMN, y=col,
        ax=ax, palette=['#3a86ff', '#fb5607'],
    )
    ax.set_title(col)
    ax.set_xticklabels(['No disease', 'Disease'])
fig.suptitle('Numeric feature spread by class', y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'boxplots_by_target.png')
plt.show()

## 8. Pairplot for the top-correlated subset

In [ ]:
top_features = (
    corr[TARGET_COLUMN].drop(TARGET_COLUMN).abs().sort_values(ascending=False).head(4).index.tolist()
)
g = sns.pairplot(
    df[top_features + [TARGET_COLUMN]],
    hue=TARGET_COLUMN, palette=['#3a86ff', '#fb5607'],
    diag_kind='kde', plot_kws={'alpha': 0.6},
)
g.fig.suptitle('Pairplot of top-correlated features', y=1.02)
g.fig.savefig(PLOTS_DIR / 'pairplot_top_features.png')
plt.show()
print('Top features:', top_features)

## 9. EDA Findings & Modelling Implications

1. **Sample size**: 303 rows — small enough that we must use cross-validation and stratified splits.
2. **Class balance**: ~54% positive / 46% negative — mild imbalance; accuracy + ROC-AUC are both informative metrics.
3. **Missingness**: only `ca` and `thal` carry a small number of missing values; median / most-frequent imputation is sufficient.
4. **Mixed feature types**: numeric (`age`, `trestbps`, `chol`, `thalach`, `oldpeak`, `ca`) require scaling; categorical (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal`) require one-hot encoding.
5. **Top predictive correlations** (|corr| with target) typically include `oldpeak`, `thalach` and `age`, which agree with clinical literature.
6. **Outliers**: `chol` shows extreme values; tree-based models (Random Forest, Gradient Boosting) handle them better than Logistic Regression — a good reason to compare both.

These observations directly drive the design of `src/preprocessing.py` and the model catalogue in `src/train.py`.